# Final Model Training

This notebook trains the final classical models on the complete development dataset using the 30-feature modality-balanced set selected after evaluation in Notebook 2. The decision threshold for each model is derived from the participant-wise nested-CV thresholds, and the trained pipelines are saved with their feature lists and metadata.

## 1. Setup

In [ ]:
import joblib
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Load Final Training Inputs

The cleaned feature dataset, final 30-feature list, and fold-specific nested-CV thresholds are loaded from `data/`. The assertions ensure that the expected feature set is available before model fitting begins.

In [ ]:
from pathlib import Path
import pandas as pd

# Google Drive paths
DATA_DIR = Path("/content/drive/MyDrive/data")
MODEL_DIR = Path("/content/drive/MyDrive/models")

# Data and artifacts produced by the previous notebooks
data_path = DATA_DIR / "stress_features_30s_multimodal_cleaned.csv"
feature_path = DATA_DIR / "final_balanced_top30_features.csv"
threshold_path = DATA_DIR / "nested_thresholds_by_fold.csv"

# Load final dataset and Notebook 2 artifacts
df = pd.read_csv(data_path)
final_features = pd.read_csv(feature_path)["feature"].tolist()
thresholds_by_fold = pd.read_csv(threshold_path)

print("Dataset shape:", df.shape)
print("Participants:", df["participant_id"].nunique())

print("\nClass distribution:")
print(df["label"].value_counts())

print("\nFinal number of features:", len(final_features))

# Sanity checks
assert len(final_features) == 30, (
    f"Expected 30 final features, found {len(final_features)}."
)

missing_features = [
    feature
    for feature in final_features
    if feature not in df.columns
]

assert not missing_features, (
    f"Features missing from dataset: {missing_features}"
)

print("\nFinal balanced top-30 features:")
for i, feature in enumerate(final_features, start=1):
    print(f"{i:2d}. {feature}")

print("\nNested-CV thresholds:")
display(thresholds_by_fold)

Dataset shape: (4540, 157)
Participants: 35

Class distribution:
label
REST      3902
STRESS     638
Name: count, dtype: int64

Final number of features: 30

Final balanced top-30 features:
 1. SCR_Peaks_N_DeltaBaseline
 2. EDA_Raw_Max
 3. EDA_Raw_Mean_DeltaBaseline
 4. SCR_Amplitude_Mean_DeltaBaseline
 5. EDA_Tonic_Slope
 6. EDA_Phasic_SD
 7. EDA_Raw_SD
 8. SCR_RiseTime_Mean
 9. SCR_Peaks_N
10. SCR_Amplitude_Sum
11. PPG_Rate_Mean_DeltaBaseline
12. PPG_Rate_Mean
13. BVP_Clean_Median
14. HRV_pNN50
15. PPI_Min
16. HRV_pNN20
17. BVP_Clean_SD
18. HRV_RMSSD_DeltaBaseline
19. BVP_Deriv2_SD
20. PPG_Rate_SD
21. ACC_Mag_P95
22. ACC_Mag_Median
23. ACC_Mag_P75
24. ACC_Mag_P25
25. ACC_Z_Mean
26. ACC_Y_Mean
27. TEMP_Mean_DeltaBaseline
28. TEMP_Median
29. TEMP_SD
30. TEMP_Slope

Nested-CV thresholds:


,fold,model,threshold
0,1,ExtraTrees,0.25
1,2,ExtraTrees,0.25
2,3,ExtraTrees,0.35
3,4,ExtraTrees,0.30
4,5,ExtraTrees,0.25
5,1,Logistic Regression,0.50
6,2,Logistic Regression,0.45
7,3,Logistic Regression,0.45
8,4,Logistic Regression,0.50
9,5,Logistic Regression,0.70


## 3. Final Decision Thresholds

For each retained model, the deployment threshold is the median of the thresholds selected by inner cross-validation across the five outer participant-wise folds.

In [ ]:
# LABEL
y_final = df["label"].map({
    "REST": 0,
    "STRESS": 1
})

# FINAL THRESHOLDS
final_thresholds = {

    "Logistic Regression": float(
        thresholds_by_fold.loc[
            thresholds_by_fold["model"]
            == "Logistic Regression",
            "threshold"
        ].median()
    ),

    "Random Forest": float(
        thresholds_by_fold.loc[
            thresholds_by_fold["model"]
            == "Random Forest",
            "threshold"
        ].median()
    ),

    "XGBoost": float(
        thresholds_by_fold.loc[
            thresholds_by_fold["model"]
            == "XGBoost",
            "threshold"
        ].median()
    )
}


print("Final thresholds:")

for model_name, threshold in (
    final_thresholds.items()
):
    print(
        f"{model_name}: "
        f"{threshold:.2f}"
    )

Final thresholds:
Logistic Regression: 0.50
Random Forest: 0.25
XGBoost: 0.25


## 4. Final Model Definitions

In [ ]:
# FINAL MODEL DEFINITIONS
def make_logistic_regression():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "clf",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                solver="liblinear",
                random_state=42
            )
        )
    ])


def make_random_forest():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "clf",
            RandomForestClassifier(
                n_estimators=500,
                class_weight=
                    "balanced_subsample",
                min_samples_leaf=3,
                random_state=42,
                n_jobs=-1
            )
        )
    ])


def make_xgboost(y_train):

    # Class imbalance is calculated
    # using the FINAL training dataset.
    n_rest = int(
        (y_train == 0).sum()
    )

    n_stress = int(
        (y_train == 1).sum()
    )

    scale_pos_weight = (
        n_rest / n_stress
    )

    print(
        "XGBoost scale_pos_weight:",
        scale_pos_weight
    )

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "clf",
            XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=3,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=
                    scale_pos_weight,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

In [ ]:
final_models = {

    "Logistic Regression":
        make_logistic_regression(),

    "Random Forest":
        make_random_forest(),

    "XGBoost":
        make_xgboost(
            y_final
        )
}


final_save_names = {

    "Logistic Regression":
        "final_logreg_balanced_top30",

    "Random Forest":
        "final_rf_balanced_top30",

    "XGBoost":
        "final_xgboost_balanced_top30"
}


print(
    "Final models:",
    list(final_models.keys())
)

XGBoost scale_pos_weight: 6.115987460815047
Final models: ['Logistic Regression', 'Random Forest', 'XGBoost']


## 5. Train and Save Final Models

Each model is fitted on all available development windows using the final 30 features. The saved artifacts include the complete preprocessing/model pipeline, ordered feature list, decision threshold, label mapping, and training metadata.

In [ ]:
# FINAL TRAINING ON ALL 35 PARTICIPANTS

model_dir = MODEL_DIR
model_dir.mkdir(parents=True, exist_ok=True)

X_final = df[final_features].copy()

saved_final_models = {}

for model_name, model in final_models.items():

    threshold = final_thresholds[model_name]
    save_name = final_save_names[model_name]

    print("\n" + "=" * 80)
    print("FINAL MODEL:", model_name)
    print("=" * 80)

    print("Participants:", df["participant_id"].nunique())
    print("Windows:", len(df))
    print("Features:", len(final_features))
    print("Threshold:", threshold)

    # TRAIN
    model.fit(X_final, y_final)

    # SAVE PATHS
    pipeline_path = model_dir / f"{save_name}_pipeline.joblib"
    features_path = model_dir / f"{save_name}_features.csv"
    metadata_path = model_dir / f"{save_name}_metadata.json"

    # SAVE PIPELINE
    joblib.dump(model, pipeline_path)

    # SAVE FEATURE LIST
    pd.Series(final_features).to_csv(
        features_path,
        index=False,
        header=["feature"]
    )

    # METADATA
    metadata = {
        "model_name": model_name,

        "feature_set": "final_modality_balanced_top30",

        "n_features": len(final_features),

        "threshold": float(threshold),

        "threshold_strategy":
            "Median of inner-CV-selected thresholds "
            "across five outer folds",

        "feature_selection":
            "Modality-balanced top-30 selected on the full "
            "development dataset after model evaluation",

        "feature_selection_quotas": {
            "EDA": 10,
            "PPG_HRV": 10,
            "ACC": 6,
            "TEMP": 4
        },

        "label_mapping": {
            "REST": 0,
            "STRESS": 1
        },

        "positive_class": "STRESS",

        "training_participants": int(
            df["participant_id"].nunique()
        ),

        "training_windows": int(len(df)),

        "rest_windows": int(
            (y_final == 0).sum()
        ),

        "stress_windows": int(
            (y_final == 1).sum()
        ),

        "window_sec": 30,

        "overlap_sec": 15,

        "evaluation":
            "Five-fold outer participant-wise GroupKFold "
            "with train-only feature selection and inner "
            "participant-wise threshold selection",

        "performance_source": "nested_cv_summary.csv",

        # Store only filenames so metadata is independent
        # of your local Google Drive path
        "pipeline_file": pipeline_path.name,

        "features_file": features_path.name
    }

    with open(metadata_path, "w") as f:
        json.dump(
            metadata,
            f,
            indent=4
        )

    saved_final_models[model_name] = {
        "model": model,
        "features": final_features.copy(),
        "threshold": threshold,
        "metadata": metadata
    }

    print("Saved:", pipeline_path)
    print("Saved:", features_path)
    print("Saved:", metadata_path)


FINAL MODEL: Logistic Regression
Participants: 35
Windows: 4540
Features: 30
Threshold: 0.5
Saved: /content/drive/MyDrive/models/final_logreg_balanced_top30_pipeline.joblib
Saved: /content/drive/MyDrive/models/final_logreg_balanced_top30_features.csv
Saved: /content/drive/MyDrive/models/final_logreg_balanced_top30_metadata.json

FINAL MODEL: Random Forest
Participants: 35
Windows: 4540
Features: 30
Threshold: 0.25
Saved: /content/drive/MyDrive/models/final_rf_balanced_top30_pipeline.joblib
Saved: /content/drive/MyDrive/models/final_rf_balanced_top30_features.csv
Saved: /content/drive/MyDrive/models/final_rf_balanced_top30_metadata.json

FINAL MODEL: XGBoost
Participants: 35
Windows: 4540
Features: 30
Threshold: 0.25
Saved: /content/drive/MyDrive/models/final_xgboost_balanced_top30_pipeline.joblib
Saved: /content/drive/MyDrive/models/final_xgboost_balanced_top30_features.csv
Saved: /content/drive/MyDrive/models/final_xgboost_balanced_top30_metadata.json


## 6. Final Model Interpretation

The following tables summarize model-specific feature contributions for inspection. They describe models fitted on the full development dataset and are not additional held-out performance estimates.

### 6.1 Logistic Regression coefficients

In [ ]:
# LOGISTIC REGRESSION COEFFICIENTS

logreg_model = (
    saved_final_models[
        "Logistic Regression"
    ]["model"]
)

logreg_clf = (
    logreg_model.named_steps[
        "clf"
    ]
)


logreg_coef_df = pd.DataFrame({

    "feature":
        final_features,

    "coefficient":
        logreg_clf.coef_[0],

    "abs_coefficient":
        np.abs(
            logreg_clf.coef_[0]
        )

}).sort_values(
    "abs_coefficient",
    ascending=False
)


display(
    logreg_coef_df
)


logreg_coef_df.to_csv(
    DATA_DIR / "final_logreg_balanced_top30_coefficients.csv",
    index=False
)

,feature,coefficient,abs_coefficient
12,BVP_Clean_Median,-0.992974,0.992974
0,SCR_Peaks_N_DeltaBaseline,-0.767729,0.767729
24,ACC_Z_Mean,0.624709,0.624709
10,PPG_Rate_Mean_DeltaBaseline,0.538908,0.538908
14,PPI_Min,0.517038,0.517038
20,ACC_Mag_P95,-0.507855,0.507855
27,TEMP_Median,0.490476,0.490476
11,PPG_Rate_Mean,0.482486,0.482486
16,BVP_Clean_SD,0.449500,0.449500
2,EDA_Raw_Mean_DeltaBaseline,0.414289,0.414289


### 6.2 Random Forest feature importance

In [ ]:
# RANDOM FOREST FEATURE IMPORTANCE

rf_model = (
    saved_final_models[
        "Random Forest"
    ]["model"]
)

rf_clf = (
    rf_model.named_steps[
        "clf"
    ]
)


rf_importance_df = pd.DataFrame({

    "feature":
        final_features,

    "importance":
        rf_clf.feature_importances_

}).sort_values(
    "importance",
    ascending=False
)


display(
    rf_importance_df
)


rf_importance_df.to_csv(
    DATA_DIR / "final_rf_balanced_top30_feature_importance.csv",
    index=False
)

,feature,importance
10,PPG_Rate_Mean_DeltaBaseline,0.106474
4,EDA_Tonic_Slope,0.065866
11,PPG_Rate_Mean,0.052206
12,BVP_Clean_Median,0.045695
1,EDA_Raw_Max,0.044066
2,EDA_Raw_Mean_DeltaBaseline,0.043058
3,SCR_Amplitude_Mean_DeltaBaseline,0.041970
26,TEMP_Mean_DeltaBaseline,0.039205
5,EDA_Phasic_SD,0.037397
24,ACC_Z_Mean,0.036553


### 6.3 XGBoost feature importance

In [ ]:
# XGBOOST FEATURE IMPORTANCE

xgb_model = (
    saved_final_models[
        "XGBoost"
    ]["model"]
)

xgb_clf = (
    xgb_model.named_steps[
        "clf"
    ]
)


xgb_importance_df = pd.DataFrame({

    "feature":
        final_features,

    "importance":
        xgb_clf.feature_importances_

}).sort_values(
    "importance",
    ascending=False
)


display(
    xgb_importance_df
)


xgb_importance_df.to_csv(
    DATA_DIR / "final_xgboost_balanced_top30_feature_importance.csv",
    index=False
)

,feature,importance
10,PPG_Rate_Mean_DeltaBaseline,0.096403
4,EDA_Tonic_Slope,0.060175
5,EDA_Phasic_SD,0.052485
12,BVP_Clean_Median,0.050778
2,EDA_Raw_Mean_DeltaBaseline,0.040240
17,HRV_RMSSD_DeltaBaseline,0.037242
3,SCR_Amplitude_Mean_DeltaBaseline,0.036348
8,SCR_Peaks_N,0.035930
1,EDA_Raw_Max,0.034308
24,ACC_Z_Mean,0.033348


## 7. Saved-Model Sanity Check

The final cell reloads each saved pipeline, feature list, and metadata file and verifies that inference runs successfully. Predictions on the training data are shown only as a **sanity check** and must not be interpreted as a performance estimate.

In [ ]:
# SAVED MODEL SANITY CHECK

for model_name, save_name in (
    final_save_names.items()
):

    pipeline_path = (
        model_dir /
        f"{save_name}_pipeline.joblib"
    )

    features_path = (
        model_dir /
        f"{save_name}_features.csv"
    )

    metadata_path = (
        model_dir /
        f"{save_name}_metadata.json"
    )


    loaded_model = joblib.load(
        pipeline_path
    )

    loaded_features = (
        pd.read_csv(
            features_path
        )["feature"].tolist()
    )


    with open(
        metadata_path,
        "r"
    ) as f:

        loaded_metadata = (
            json.load(f)
        )


    X_check = df[
        loaded_features
    ]


    stress_scores = (
        loaded_model.predict_proba(
            X_check
        )[:, 1]
    )


    predictions = (
        stress_scores
        >= loaded_metadata[
            "threshold"
        ]
    ).astype(int)


    print(
        "\n" + "=" * 80
    )

    print(model_name)

    print(
        "Loaded features:",
        len(loaded_features)
    )

    print(
        "Loaded threshold:",
        loaded_metadata[
            "threshold"
        ]
    )

    print(
        "Predicted REST:",
        int(
            (predictions == 0).sum()
        )
    )

    print(
        "Predicted STRESS:",
        int(
            (predictions == 1).sum()
        )
    )

    print(
        "Sanity check only — "
        "NOT a performance estimate."
    )


Logistic Regression
Loaded features: 30
Loaded threshold: 0.5
Predicted REST: 3108
Predicted STRESS: 1432
Sanity check only — NOT a performance estimate.

Random Forest
Loaded features: 30
Loaded threshold: 0.25
Predicted REST: 3804
Predicted STRESS: 736
Sanity check only — NOT a performance estimate.

XGBoost
Loaded features: 30
Loaded threshold: 0.25
Predicted REST: 3103
Predicted STRESS: 1437
Sanity check only — NOT a performance estimate.
